# exp002 - visual clip baseline

this is a reconstruction of the final exp002 setup rather than the literal notebook that originally ran. the old saved notebook snapshot was stale and did not match the final archived convnext-base setup, so i used the final config, split manifests and saved results as the reference.

main setup to remember:
- AV-Deepfake1M++ usable validation subset
- exact saved group-disjoint split: 28k / 6k / 6k
- frozen imagenet convnext-base
- 16 uniformly sampled frames
- mean pooled 1024-d clip feature
- layernorm + linear real/fake classifier
- best checkpoint chosen using validation f1
- normal two-class argmax for the final prediction

## setup

seed is fixed for the classifier part. feature extraction should also be stable for the same videos/frame positions, although gpu/library differences can still give small numerical differences if this is ever rerun.

In [ ]:
# imports, seed + device setup
from pathlib import Path
from datetime import datetime
import json
import math
import random
import shutil
import warnings
import zipfile

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
)

import matplotlib.pyplot as plt
from IPython.display import display, FileLink

from exp002_utils import (
    choose_manifest_dir,
    find_one_csv,
    ConvNeXtBaseClipFeatures,
    extract_split_features,
    ClipLinearClassifier,
    evaluate,
    build_archive_index,
)

warnings.filterwarnings("ignore", category=UserWarning)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# keep the seed/device behaviour explicit in case the notebook is ever rerun
torch.set_float32_matmul_precision("high")
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Torch:", torch.__version__)
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## experiment config

In [ ]:
# paths/settings used for the final reconstructed exp002 setup
PROJECT_ROOT = Path("/home/jovyan/MSC_PROJECT")

EXPERIMENT_ID = "exp002"
RUN_NAME = "exp002_visual_clip_final_reproduction"

PATH_COL = "path"
LABEL_COL = "binary_label"
CONDITION_COL = "condition"
GROUP_COL = "clip_group"

MODEL_NAME = "convnext_base"
NUM_FRAMES = 16
IMAGE_SIZE = 224
FEATURE_DIM = 1024

# feature extraction settings from the archived final config
FEATURE_BATCH_SIZE = 8
NUM_WORKERS = 2
PREFETCH_FACTOR = 1
PIN_MEMORY = False

# classifier settings from the archived final config
CLASSIFIER_BATCH_SIZE = 2048
CLASSIFIER_EPOCHS = 30
CLASSIFIER_LR = 1e-3
CLASSIFIER_WEIGHT_DECAY = 1e-4

EXPECTED_SPLIT_ROWS = {
    "train": 28000,
    "val": 6000,
    "test": 6000,
}

EXPECTED_HISTORICAL_METRICS = {'accuracy': 0.701833, 'balanced_accuracy': 0.701833, 'f1': 0.7149, 'precision': 0.684885, 'recall': 0.747667, 'auc': 0.767401}

# can point this at the exact saved manifests if auto-discovery cannot find them
SPLIT_MANIFEST_DIR = None

RUN_DIR = PROJECT_ROOT / "experiments" / RUN_NAME
MANIFEST_OUT_DIR = RUN_DIR / "manifests"
FEATURE_DIR = RUN_DIR / "features"
RESULT_DIR = RUN_DIR / "results"
PLOT_DIR = RUN_DIR / "plots"
CHECKPOINT_DIR = RUN_DIR / "checkpoints"

for d in [RUN_DIR, MANIFEST_OUT_DIR, FEATURE_DIR, RESULT_DIR, PLOT_DIR, CHECKPOINT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "experiment_id": EXPERIMENT_ID,
    "run_name": RUN_NAME,
    "status": "reconstructed_final_reproduction_notebook",
    "historical_notebook_status": "stale_not_used_as_authoritative_execution_evidence",
    "dataset": "AV-Deepfake1M++ validation subset",
    "task": "visual_only_binary_classification",
    "target": LABEL_COL,
    "split_type": "exact_archived_clip_group_disjoint_manifests",
    "model": "Frozen ConvNeXt-Base encoder + LayerNorm/Linear classifier",
    "num_frames": NUM_FRAMES,
    "image_size": IMAGE_SIZE,
    "feature_dim": FEATURE_DIM,
    "feature_batch_size": FEATURE_BATCH_SIZE,
    "num_workers": NUM_WORKERS,
    "prefetch_factor": PREFETCH_FACTOR,
    "pin_memory": PIN_MEMORY,
    "classifier_batch_size": CLASSIFIER_BATCH_SIZE,
    "classifier_epochs": CLASSIFIER_EPOCHS,
    "classifier_lr": CLASSIFIER_LR,
    "classifier_weight_decay": CLASSIFIER_WEIGHT_DECAY,
    "checkpoint_selection": "best validation F1",
    "test_threshold": 0.5,
    "seed": SEED,
}

with open(RUN_DIR / "config.json", "w") as f:
    json.dump(CONFIG, f, indent=2)

CONFIG

## saved train / val / test split

important bit here is that i reuse the exact archived manifests rather than generating another random split. a different split would technically be a different experiment.

In [ ]:
# possible locations of the saved exp002 manifests
CANDIDATE_MANIFEST_DIRS = [
    PROJECT_ROOT / "experiments" / "exp002_visual_clip" / "manifests",
    PROJECT_ROOT / "experiments" / "002_visual_convnext_base_frozen_features_linear_group_disjoint_balanced40k_16frames" / "manifests",
    PROJECT_ROOT / "supporting_material" / "experiments" / "exp002_visual_clip" / "manifests"
]

# find the exact saved split files - helper deliberately stops instead of guessing
SOURCE_MANIFEST_DIR = choose_manifest_dir(
    SPLIT_MANIFEST_DIR,
    CANDIDATE_MANIFEST_DIRS,
)

train_csv = find_one_csv(SOURCE_MANIFEST_DIR, "train")
val_csv = find_one_csv(SOURCE_MANIFEST_DIR, "val")
test_csv = find_one_csv(SOURCE_MANIFEST_DIR, "test")

print("Using exact archived manifests from:", SOURCE_MANIFEST_DIR)
print("Train:", train_csv.name)
print("Val:  ", val_csv.name)
print("Test: ", test_csv.name)


In [ ]:
# load the saved splits and check their columns, row counts and video paths
train_df = pd.read_csv(train_csv).reset_index(drop=True)
val_df = pd.read_csv(val_csv).reset_index(drop=True)
test_df = pd.read_csv(test_csv).reset_index(drop=True)

splits = {"train": train_df, "val": val_df, "test": test_df}

for split_name, frame in splits.items():
    missing = {PATH_COL, LABEL_COL, CONDITION_COL} - set(frame.columns)
    if missing:
        raise ValueError(f"{split_name} manifest is missing columns: {missing}")

    expected_n = EXPECTED_SPLIT_ROWS[split_name]
    assert len(frame) == expected_n, (
        f"{split_name}: expected {expected_n} rows from the final run, "
        f"got {len(frame)}"
    )

    missing_paths = (~frame[PATH_COL].map(lambda x: Path(x).is_file())).sum()
    if missing_paths:
        raise FileNotFoundError(
            f"{split_name}: {missing_paths} video paths do not currently exist. "
            "Update the path column or mount the dataset before reproducing features."
        )

print("Split sizes:", {k: len(v) for k, v in splits.items()})
print("\nTrain labels:\n", train_df[LABEL_COL].value_counts().sort_index())
print("\nValidation labels:\n", val_df[LABEL_COL].value_counts().sort_index())
print("\nTest labels:\n", test_df[LABEL_COL].value_counts().sort_index())
print("\nTest conditions:\n", test_df[CONDITION_COL].value_counts())

In [ ]:
# check that no clip_group appears across train/val/test
if all(GROUP_COL in frame.columns for frame in splits.values()):
    train_groups = set(train_df[GROUP_COL].astype(str))
    val_groups = set(val_df[GROUP_COL].astype(str))
    test_groups = set(test_df[GROUP_COL].astype(str))

    overlaps = {
        "train_val": len(train_groups & val_groups),
        "train_test": len(train_groups & test_groups),
        "val_test": len(val_groups & test_groups),
    }
    print("Group overlaps:", overlaps)
    assert overlaps == {"train_val": 0, "train_test": 0, "val_test": 0}
else:
    print("clip_group is not present in every saved split; overlap cannot be re-audited here.")

In [ ]:
# keep a copy of the exact manifests alongside this run
for split_name, src in [("train", train_csv), ("val", val_csv), ("test", test_csv)]:
    shutil.copy2(src, MANIFEST_OUT_DIR / f"{split_name}.csv")
print("Copied exact split manifests to:", MANIFEST_OUT_DIR)

## frame sampling + frozen visual features

each video is represented by 16 positions sampled across the clip. the longer opencv/preprocessing code is in `exp002_utils.py`.

convnext-base is frozen and used as a feature extractor. each sampled frame gives a 1024-d feature and the 16 frame features are mean pooled into one clip vector.

In [ ]:
# create the frozen convnext-base feature extractor
feature_extractor = ConvNeXtBaseClipFeatures().to(DEVICE).eval()

print(
    "Trainable encoder parameters:",
    sum(
        p.numel()
        for p in feature_extractor.parameters()
        if p.requires_grad
    ),
)


## feature cache

video decoding + convnext inference is the expensive part, so the 1024-d clip features are cached before training the small classifier. if a complete cache already exists it can be reused.

In [ ]:
# extract/cache the visual features for each saved split
feature_args = dict(
    feature_extractor=feature_extractor,
    feature_dir=FEATURE_DIR,
    device=DEVICE,
    num_frames=NUM_FRAMES,
    image_size=IMAGE_SIZE,
    feature_dim=FEATURE_DIM,
    feature_batch_size=FEATURE_BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    prefetch_factor=PREFETCH_FACTOR,
    path_col=PATH_COL,
    label_col=LABEL_COL,
)

train_cache = extract_split_features(
    "train",
    train_df,
    **feature_args,
)

val_cache = extract_split_features(
    "val",
    val_df,
    **feature_args,
)

test_cache = extract_split_features(
    "test",
    test_df,
    **feature_args,
)


## quick feature sanity check

In [ ]:
# check the cached tensors have the expected shape and no nan/inf values
for split_name, payload in [
    ("train", train_cache),
    ("val", val_cache),
    ("test", test_cache),
]:
    X = payload["X"]
    print(
        split_name,
        "shape=", tuple(X.shape),
        "mean=", float(X.mean()),
        "std=", float(X.std()),
        "finite=", bool(torch.isfinite(X).all()),
    )
    assert torch.isfinite(X).all()
    assert X.std() > 0

## classifier

only the small classifier is trained here. the frozen clip features go through layernorm and then a linear 1024 -> 2 layer.

30 epochs, adamw, best validation f1 checkpoint.

In [ ]:
# turn the cached clip features into train/val/test loaders
X_train, y_train = train_cache["X"].float(), train_cache["y"].long()
X_val, y_val = val_cache["X"].float(), val_cache["y"].long()
X_test, y_test = test_cache["X"].float(), test_cache["y"].long()

train_loader = DataLoader(
    TensorDataset(X_train, y_train),
    batch_size=CLASSIFIER_BATCH_SIZE,
    shuffle=True,
    generator=torch.Generator().manual_seed(SEED),
)
val_loader = DataLoader(
    TensorDataset(X_val, y_val),
    batch_size=CLASSIFIER_BATCH_SIZE,
    shuffle=False,
)
test_loader = DataLoader(
    TensorDataset(X_test, y_test),
    batch_size=CLASSIFIER_BATCH_SIZE,
    shuffle=False,
)

# classifier itself is defined in exp002_utils.py
classifier = ClipLinearClassifier(FEATURE_DIM).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    classifier.parameters(),
    lr=CLASSIFIER_LR,
    weight_decay=CLASSIFIER_WEIGHT_DECAY,
)

classifier

In [ ]:
# actual classifier training loop - save whichever epoch gives the best val f1
best_val_f1 = -np.inf
history = []
checkpoint_path = CHECKPOINT_DIR / "best_classifier.pt"

for epoch in range(1, CLASSIFIER_EPOCHS + 1):
    classifier.train()
    train_loss = 0.0
    train_true, train_pred = [], []

    for X, y in train_loader:
        X = X.to(DEVICE)
        y = y.to(DEVICE)

        optimizer.zero_grad(set_to_none=True)
        logits = classifier(X)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        pred = logits.argmax(dim=1)
        train_loss += loss.item() * len(y)
        train_true.extend(y.detach().cpu().numpy())
        train_pred.extend(pred.detach().cpu().numpy())

    train_metrics = {
        "loss": train_loss / len(train_loader.dataset),
        "accuracy": accuracy_score(train_true, train_pred),
        "balanced_accuracy": balanced_accuracy_score(train_true, train_pred),
        "f1": f1_score(train_true, train_pred, zero_division=0),
        "precision": precision_score(train_true, train_pred, zero_division=0),
        "recall": recall_score(train_true, train_pred, zero_division=0),
    }

    val_metrics, _, _, _ = evaluate(classifier, val_loader, criterion, DEVICE)

    row = {
        "epoch": epoch,
        **{f"train_{k}": v for k, v in train_metrics.items()},
        **{f"val_{k}": v for k, v in val_metrics.items()},
    }
    history.append(row)
    pd.DataFrame(history).to_csv(RESULT_DIR / "training_history.csv", index=False)

    print(
        f"epoch={epoch:02d} "
        f"train_f1={train_metrics['f1']:.4f} "
        f"val_f1={val_metrics['f1']:.4f} "
        f"val_auc={val_metrics['auc']:.4f}"
    )

    if val_metrics["f1"] > best_val_f1:
        best_val_f1 = val_metrics["f1"]
        torch.save(
            {
                "model_state_dict": classifier.state_dict(),
                "epoch": epoch,
                "val_metrics": val_metrics,
                "feature_dim": FEATURE_DIM,
                "num_frames": NUM_FRAMES,
                "seed": SEED,
            },
            checkpoint_path,
        )

print("Best validation F1:", best_val_f1)
print("Checkpoint:", checkpoint_path)

## final test

In [ ]:
# reload the best validation checkpoint, then evaluate once on the held-out test split
ckpt = torch.load(checkpoint_path, map_location=DEVICE)
classifier.load_state_dict(ckpt["model_state_dict"])

test_metrics, y_true, y_pred, y_prob = evaluate(classifier, test_loader, criterion, DEVICE)
display(pd.DataFrame([test_metrics]))
pd.DataFrame([test_metrics]).to_csv(RESULT_DIR / "test_metrics.csv", index=False)

test_predictions = test_df.copy()
test_predictions["y_true"] = y_true
test_predictions["y_pred"] = y_pred
test_predictions["fake_probability"] = y_prob
test_predictions.to_csv(RESULT_DIR / "test_predictions.csv", index=False)

## condition-level check

In [ ]:
# check how often each manipulation condition is being called fake
rows = []
for condition, group in test_predictions.groupby(CONDITION_COL):
    true_values = sorted(group["y_true"].unique().tolist())
    pred_fake_rate = float(group["y_pred"].mean())
    mean_prob = float(group["fake_probability"].mean())

    if len(true_values) == 1 and int(true_values[0]) == 0:
        success = 1.0 - pred_fake_rate
        interpretation = "real specificity"
    elif len(true_values) == 1 and int(true_values[0]) == 1:
        success = pred_fake_rate
        interpretation = "fake recall"
    else:
        success = float((group["y_true"] == group["y_pred"]).mean())
        interpretation = "accuracy"

    rows.append({
        "condition": condition,
        "n": len(group),
        "true_labels": str(true_values),
        "condition_success_rate": success,
        "interpretation": interpretation,
        "predicted_fake_rate": pred_fake_rate,
        "mean_fake_probability": mean_prob,
    })

condition_metrics = pd.DataFrame(rows).sort_values("condition")
display(condition_metrics)
condition_metrics.to_csv(RESULT_DIR / "condition_metrics.csv", index=False)

saved historical condition results for comparison if this reconstruction is ever rerun:

| condition | n | final success rate |
|---|---:|---:|
| `fake_video_fake_audio` | 1001 | 0.753247 fake recall |
| `fake_video_real_audio` | 972 | 0.748971 fake recall |
| `real` | 3000 | 0.656000 specificity |
| `real_video_fake_audio` | 1027 | 0.740993 fake recall |

## plots

In [ ]:
# confusion matrix, roc curve and classifier f1 history
fig, ax = plt.subplots(figsize=(6, 6))
ConfusionMatrixDisplay.from_predictions(
    y_true, y_pred,
    display_labels=["Real", "Fake"],
    values_format="d",
    ax=ax,
)
ax.set_title(f"{RUN_NAME}: test confusion matrix")
fig.tight_layout()
fig.savefig(PLOT_DIR / "confusion_matrix.png", dpi=200)
plt.show()

fig, ax = plt.subplots(figsize=(6, 6))
RocCurveDisplay.from_predictions(y_true, y_prob, ax=ax)
ax.set_title(f"{RUN_NAME}: test ROC")
fig.tight_layout()
fig.savefig(PLOT_DIR / "roc_curve.png", dpi=200)
plt.show()

hist = pd.read_csv(RESULT_DIR / "training_history.csv")
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(hist["epoch"], hist["train_f1"], label="train_f1")
ax.plot(hist["epoch"], hist["val_f1"], label="val_f1")
ax.set_xlabel("Epoch")
ax.set_ylabel("F1")
ax.set_title(f"{RUN_NAME}: classifier training")
ax.legend()
fig.tight_layout()
fig.savefig(PLOT_DIR / "training_f1.png", dpi=200)
plt.show()

## compare against the archived result

this is only a check for the reconstructed notebook. the archived final metrics are the values used in the dissertation, so a rerun should be compared against them rather than silently replacing them.

In [ ]:
# compare each reproduced test metric with the saved historical value
comparison_rows = []
for metric, historical_value in EXPECTED_HISTORICAL_METRICS.items():
    reproduced_value = float(test_metrics[metric])
    comparison_rows.append({
        "metric": metric,
        "historical_final": historical_value,
        "reproduced": reproduced_value,
        "absolute_difference": abs(reproduced_value - historical_value),
    })

comparison = pd.DataFrame(comparison_rows)
display(comparison)
comparison.to_csv(RESULT_DIR / "historical_vs_reproduced_metrics.csv", index=False)

## package the reconstruction

cached feature tensors are left out of the zip because they are large and can be regenerated from the saved manifests + raw dataset.

In [ ]:
# index the files in the run folder, then create a smaller export zip
display(build_archive_index(RUN_DIR))

EXPORT_DIR = PROJECT_ROOT / "exports"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
zip_path = EXPORT_DIR / f"{RUN_NAME}_{stamp}.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in RUN_DIR.rglob("*"):
        if not p.is_file():
            continue
        rel = p.relative_to(RUN_DIR)
        if "features" in rel.parts and p.suffix == ".pt":
            continue
        zf.write(p, arcname=(RUN_DIR.name / rel) if False else str(Path(RUN_DIR.name) / rel))

print("Created:", zip_path)
display(FileLink(str(zip_path)))

## interpretation note

this is a visual-only **any-fake** baseline.

the slightly weird result to remember is that a visually real / audio-fake clip can still be predicted as fake by the visual branch. that does **not** mean the visual model somehow detected the fake audio. it can also come from visual or production-related shortcut cues in that condition.

the source split is clip-group-disjoint, but it is not guaranteed to be identity-disjoint.